# 로지스틱 회귀 모델링 및 교차검증

`quarter` 컬럼을 제거하고 `posting_month`를 유지한 월별 데이터셋(`jswBad_1_monthly.csv`)을 사용해 `high_salary` 수요예측 분류 모델을 만듭니다. 다중공선성 가능성을 고려하여 기본 모델에서는 `experience_level` 컬럼을 제외합니다.

## 1. 라이브러리 불러오기

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", font="AppleGothic")
plt.rcParams["axes.unicode_minus"] = False

RANDOM_STATE = 42

## 2. 데이터 불러오기

In [ ]:
candidate_paths = [
    Path("jswBad_1_monthly.csv"),
    Path("수요예측_모델검증/jswBad_1_monthly.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("jswBad_1_monthly.csv 파일을 현재 폴더 또는 수요예측_모델검증 폴더에서 찾을 수 없습니다.")

df = pd.read_csv(data_path)

print(f"데이터 경로: {data_path}")
print(f"데이터 크기: {df.shape[0]:,}행 x {df.shape[1]:,}열")
df.head()

## 3. 데이터 기본 확인

In [ ]:
print("quarter 컬럼 포함 여부:", "quarter" in df.columns)
print("posting_month 컬럼 포함 여부:", "posting_month" in df.columns)
print("결측치 총 개수:", int(df.isna().sum().sum()))
print("중복 행 개수:", int(df.duplicated().sum()))

target_counts = df["high_salary"].value_counts().sort_index()
target_ratio = df["high_salary"].value_counts(normalize=True).sort_index()

pd.DataFrame({
    "count": target_counts,
    "ratio": target_ratio,
})

## 4. 독립변수와 종속변수 분리

In [ ]:
target_col = "high_salary"

drop_cols = [target_col, "experience_level"]

X = df.drop(columns=drop_cols)
y = df[target_col]

print(f"X 크기: {X.shape}")
print(f"y 크기: {y.shape}")
print("제외한 컬럼:", drop_cols[1:])
print("타깃 비율")
print(y.value_counts(normalize=True).sort_index())

## 5. 학습용/검증용 데이터 분리

`high_salary`의 0/1 비율을 유지하기 위해 `stratify=y`를 적용합니다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"학습 데이터: {X_train.shape}")
print(f"검증 데이터: {X_test.shape}")
print("학습 데이터 타깃 비율")
print(y_train.value_counts(normalize=True).sort_index())
print("검증 데이터 타깃 비율")
print(y_test.value_counts(normalize=True).sort_index())

## 6. 로지스틱 회귀 모델 학습

데이터가 이미 전처리된 형태지만, 연속형 변수와 0/1 변수가 함께 있으므로 `StandardScaler`와 `LogisticRegression`을 파이프라인으로 묶어 사용합니다.

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                solver="lbfgs",
            ),
        ),
    ]
)

logistic_model.fit(X_train, y_train)

## 7. 검증 데이터 성능 평가

In [ ]:
y_pred = logistic_model.predict(X_test)
y_pred_proba = logistic_model.predict_proba(X_test)[:, 1]

test_metrics = pd.DataFrame(
    {
        "metric": ["accuracy", "precision", "recall", "f1", "roc_auc"],
        "score": [
            accuracy_score(y_test, y_pred),
            precision_score(y_test, y_pred),
            recall_score(y_test, y_pred),
            f1_score(y_test, y_pred),
            roc_auc_score(y_test, y_pred_proba),
        ],
    }
)

test_metrics

In [ ]:
cm = confusion_matrix(y_test, y_pred)

confusion_df = pd.DataFrame(
    cm,
    index=["actual_0", "actual_1"],
    columns=["pred_0", "pred_1"],
)

confusion_df

In [ ]:
print(classification_report(y_test, y_pred, target_names=["low_salary", "high_salary"]))

## 8. Stratified K-Fold 교차검증

타깃 클래스 비율을 각 fold에 비슷하게 유지하기 위해 `StratifiedKFold`를 사용합니다.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

cv_results = cross_validate(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

cv_scores = pd.DataFrame(cv_results)
cv_scores

In [ ]:
score_cols = [col for col in cv_scores.columns if col.startswith("test_")]

cv_summary = (
    cv_scores[score_cols]
    .agg(["mean", "std"])
    .T
    .rename_axis("metric")
    .reset_index()
)

cv_summary["metric"] = cv_summary["metric"].str.replace("test_", "", regex=False)
cv_summary

## 9. 교차검증 결과 시각화

5-Fold 교차검증 결과를 평균 ± 표준편차와 fold별 성능 변화로 확인합니다.

In [ ]:
metric_order = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
metric_labels = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"]

means = np.array([cv_scores[f"test_{metric}"].mean() * 100 for metric in metric_order])
stds = np.array([cv_scores[f"test_{metric}"].std() * 100 for metric in metric_order])

fold_scores = pd.DataFrame({"Fold": np.arange(1, len(cv_scores) + 1)})
for metric, label in zip(metric_order, metric_labels):
    fold_scores[label] = cv_scores[f"test_{metric}"] * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].bar(metric_labels, means, yerr=stds, capsize=4, color="#5b84b1", alpha=0.92)
axes[0].set_title("Stratified K-Fold 평균 ± 표준편차")
axes[0].set_xlabel("지표")
axes[0].set_ylabel("점수 (%)")
axes[0].set_ylim(0, 105)
axes[0].tick_params(axis="x", rotation=35)

for index, value in enumerate(means):
    axes[0].text(index, value + 2.0, f"{value:.1f}", ha="center", fontsize=9)

for label in metric_labels:
    axes[1].plot(fold_scores["Fold"], fold_scores[label], marker="o", linewidth=2, label=label)

axes[1].set_title("Fold별 성능 변화")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("점수 (%)")
axes[1].set_xticks(np.arange(1, len(cv_scores) + 1))
axes[1].set_ylim(0, 105)
axes[1].legend(loc="lower right")

fig.suptitle("experience_level 제거 후 Stratified 5-Fold 교차검증 결과", y=1.02)
fig.tight_layout()
plt.show()

## 10. 로지스틱 회귀 계수 확인

계수의 절댓값이 클수록 모델 판단에 더 크게 반영된 변수입니다. 계수가 양수이면 `high_salary=1` 방향, 음수이면 `high_salary=0` 방향으로 작용합니다.

In [ ]:
feature_names = X.columns
coefficients = logistic_model.named_steps["model"].coef_[0]

coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coefficients,
        "abs_coefficient": np.abs(coefficients),
    }
).sort_values("abs_coefficient", ascending=False)

coef_df.head(20)

In [ ]:
positive_features = coef_df.sort_values("coefficient", ascending=False).head(15)
negative_features = coef_df.sort_values("coefficient", ascending=True).head(15)

print("high_salary=1 방향으로 작용하는 주요 변수")
display(positive_features[["feature", "coefficient"]])

print("high_salary=0 방향으로 작용하는 주요 변수")
display(negative_features[["feature", "coefficient"]])

## 11. 보고서용 성능 시각화

검증 데이터 성능지표와 혼동행렬을 그래프로 확인합니다.

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=test_metrics, x="metric", y="score", hue="metric", palette="Blues_d", legend=False)
plt.ylim(0, 1)
plt.title("Logistic Regression Test Metrics")
plt.xlabel("Metric")
plt.ylabel("Score")

for index, row in test_metrics.iterrows():
    plt.text(index, row["score"] + 0.01, f"{row['score']:.3f}", ha="center")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_df, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()

## 12. SHAP 기반 변수 영향 시각화

SHAP 패키지가 설치되어 있으면 SHAP summary plot을 출력합니다. 설치되어 있지 않은 환경에서는 로지스틱 회귀 계수 기준 변수 중요도 그래프를 대신 출력합니다.

In [ ]:
try:
    import shap

    sample_size = min(1000, len(X_test))
    X_test_sample = X_test.sample(sample_size, random_state=RANDOM_STATE)
    X_test_sample_scaled = logistic_model.named_steps["scaler"].transform(X_test_sample)

    explainer = shap.LinearExplainer(
        logistic_model.named_steps["model"],
        X_test_sample_scaled,
        feature_names=X.columns,
    )
    shap_values = explainer(X_test_sample_scaled)

    shap.summary_plot(shap_values, X_test_sample, feature_names=X.columns, max_display=20)

except ImportError:
    print("SHAP 패키지가 설치되어 있지 않아 로지스틱 회귀 계수 기반 중요도 그래프를 출력합니다.")

    top_coef = coef_df.head(20).copy()
    top_coef = top_coef.sort_values("coefficient")
    top_coef["direction"] = np.where(top_coef["coefficient"] >= 0, "high_salary 방향", "low_salary 방향")

    plt.figure(figsize=(9, 8))
    sns.barplot(
        data=top_coef,
        x="coefficient",
        y="feature",
        hue="direction",
        palette={"high_salary 방향": "#2c7fb8", "low_salary 방향": "#d95f0e"},
    )
    plt.axvline(0, color="black", linewidth=1)
    plt.title("Top 20 Logistic Regression Coefficients")
    plt.xlabel("Coefficient")
    plt.ylabel("Feature")
    plt.legend(title="Direction")
    plt.tight_layout()
    plt.show()

## 13. 최종 모델 재학습

교차검증과 검증 데이터 평가를 확인한 뒤, 전체 데이터를 사용해 최종 모델을 다시 학습합니다.

In [ ]:
final_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                solver="lbfgs",
            ),
        ),
    ]
)

final_model.fit(X, y)

print("최종 로지스틱 회귀 모델 학습 완료")